# atlas — Colab A100 deep-dive debug session

Runs our real harness (`atlas_src`) against **2-3 specific ARC-AGI-3 games**, played to real completion (no 30-minute Kaggle calibration cap), so we can read full transcripts and understand *why* the model fails, not just *that* it fails.

Why Colab and not Kaggle: our weekly Kaggle GPU quota is tight right now, and a real submission kernel never talks to the live ARC-AGI-3 API anyway (`enable_internet` is forced off for any kernel with a competition data source — see `inference/framework/run.py:_effective_environments_dir`, which is why Kaggle plays from a bundled *offline* JSON clone of the games instead). Colab has normal internet, so it hits the **same public HTTPS API** (`https://three.arcprize.org`) directly, with `arc_agi` fetching an anonymous key when `ARC_API_KEY` isn't set — no Kaggle quota touched at all.

## Selected games and what each one tests

Picked from the real v19 Phase-A run (`runs/arc3_atlas_v19_25_08/summary.txt`), each one targeting a specific, already-diagnosed failure mode rather than just "a hard game":

1. **`cn04-2fe56bfb`** — *position-drift hypothesis*. In the Gemini-vs-Qwen comparison, the model identifies the mechanic in 1-2 turns but never converges because it re-derives the absolute position from scratch every turn instead of anchoring a confirmed one. This is exactly what the new memo/hash-anchor prompt addition (not yet shipped to Kaggle) targets — watch whether the model actually writes a confirmed position/hash into `memo` and reads it back next turn, or keeps recomputing from pixels.
2. **`vc33-5430563c`** — *stalls after a real success*. Best v19 score (10.71, level 2/7) and the most actions/tokens of any game, yet progress still stopped. Watch what happens to `memo`/the working theory exactly at the level-transition boundary: does the model try to reuse anything from level 1, or discard it and restart from zero?
3. **`ka59-38d34dbb`** — *high tokens, almost no actions* (8 actions, 37k tokens in v19). A milder cousin of the original `r11l` total-paralysis case. Watch whether the force-act / goal-reconsider checkpoints actually fire, and whether the model's behavior visibly changes in the very next turn after one fires (adoption latency), or if it just performs a token action to satisfy the checkpoint without changing its approach.

Edit `GAME_IDS` below to swap any of these (e.g. for `r11l-495a7899`, the original paralysis case, instead of `ka59`).

## Prerequisites (do these before running)

1. **A100 runtime**: Runtime -> Change runtime type -> A100 GPU. A T4/V100 cannot fit the 27B FP8 model (~27GB weights alone).
2. **Push local changes to your fork** so Colab sees the *current* code, including the memo/hash-anchor and goal-reconsider changes that haven't gone into any Kaggle kernel yet:
   ```
   git push mine main
   ```
   (remote `mine` = `https://github.com/sertru1000-cpu/ARC-AGI3.git`). If you'd rather not push yet, zip `atlas_src/src` locally and upload it instead — see the commented alternative in the clone cell.
3. **`kaggle.json`** API token (kaggle.com -> Account -> API section -> Create New API Token, or "Legacy API Credentials" -> Create Legacy API Key if that's what your account shows) — needed to download the private Qwen model.

## Known risk, stated up front

The real Kaggle deployment installs vLLM/torch/flashinfer from a private *H100* wheelhouse dataset (`driessmit1/arc3-vllm-h100-wheelhouse-v3`), not plain PyPI. This notebook installs the same pinned versions (`vllm==0.19.0`, `torch==2.10.0`, `flashinfer==0.6.6`) straight from PyPI instead, since Colab has internet and that wheelhouse isn't public. This is very likely fine (Ampere/A100 is a standard, widely-supported target), but it is **not proven** — if install or model loading fails, the fallback is to drop the version pins (`pip install vllm torch`) and let pip resolve versions compatible with Colab's actual CUDA/driver. Worth budgeting a few minutes for this on the first run.

**Confirmed 25.08 on a live Colab session**: Colab's own kernel now runs Python 3.13, but `vllm`/`flashinfer` don't ship 3.13 wheels yet and `ARC3-Inference/pyproject.toml` pins `requires-python==3.12.12` exactly. The dependency-install cell below therefore creates an isolated Python 3.12 venv with `uv` (downloads a standalone interpreter, no apt/sudo needed) and runs everything — vLLM server, harness — through that venv's binaries instead of Colab's own kernel Python. If a Colab session disconnects (idle timeout, network hiccup), `/content` is wiped; re-run from the clone cell onward rather than resuming partway.

**Compute units**: A100 costs roughly ~12 units/hour on Colab. Three games at up to 60 min each (see `MAX_RUNTIME_MINUTES` below), sequential, plus ~15-20 min one-time setup/model-load, is about 3.5-4 hours total — well inside a 100-unit pack. Stop the runtime when done (Runtime -> Manage sessions -> Terminate) rather than leaving it idle; units drain while the runtime is alive, not just while it's computing.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import subprocess

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
).stdout.strip()
assert "A100" in gpu_name, (
    f"Expected an A100 GPU, got: {gpu_name!r}. "
    "Go to Runtime -> Change runtime type -> A100 GPU, then re-run."
)
print(f"GPU OK: {gpu_name}")

In [ ]:
# Clone the harness source. Push your local atlas_src changes to your fork FIRST
# (git push mine main) so this picks up the current working tree, not a stale commit.
REPO_URL = "https://github.com/sertru1000-cpu/ARC-AGI3.git"
BRANCH = "main"
REPO_DIR = "/content/ARC-AGI-3"

!rm -rf {REPO_DIR}
!git clone --branch {BRANCH} --single-branch --depth 1 {REPO_URL} {REPO_DIR}

# --- Alternative: upload a zip of atlas_src/src instead of cloning ---
# from google.colab import files
# uploaded = files.upload()  # select an atlas_src_src.zip made locally with:
#   (cd atlas_src && zip -r ../atlas_src_src.zip src)
# import zipfile, pathlib
# pathlib.Path(f"{REPO_DIR}/atlas_src").mkdir(parents=True, exist_ok=True)
# with zipfile.ZipFile(next(iter(uploaded))) as zf:
#     zf.extractall(f"{REPO_DIR}/atlas_src")

ATLAS_SRC = f"{REPO_DIR}/atlas_src/src"

# Fail loudly here instead of a confusing "Distribution not found" three cells
# later -- a partial/failed clone (network hiccup, etc.) leaves these variables
# set even though nothing actually landed on disk.
from pathlib import Path

for _pkg in ("tufa-arc-agi-framework", "ARC3-Inference"):
    _marker = Path(ATLAS_SRC) / _pkg / "pyproject.toml"
    if not _marker.is_file():
        raise FileNotFoundError(
            f"Clone looks incomplete: {_marker} does not exist. "
            "Re-run this cell (check the git clone output above for errors)."
        )

print("ARC3-Inference:", f"{ATLAS_SRC}/ARC3-Inference")
print("tufa-arc-agi-framework:", f"{ATLAS_SRC}/tufa-arc-agi-framework")
print("Clone OK.")

In [ ]:
# Upload kaggle.json (kaggle.com -> Settings -> API -> Create New Token).
from google.colab import files
import os, shutil, stat

uploaded = files.upload()
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
src_name = next(iter(uploaded))
shutil.move(src_name, os.path.join(kaggle_dir, "kaggle.json"))
os.chmod(os.path.join(kaggle_dir, "kaggle.json"), stat.S_IRUSR | stat.S_IWUSR)

!pip install -q kagglehub

In [ ]:
# Download the private Qwen3.8-27B-FP8 repacked model (same one the real Kaggle
# submission uses). This is ~27GB+ — expect several minutes.
import kagglehub
from pathlib import Path

MODEL_HANDLE = "foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"
MODEL_PATH = Path(kagglehub.model_download(MODEL_HANDLE))
print("Model downloaded to:", MODEL_PATH)

# Same sanity check submission.ipynb runs before trusting the mount.
_required = [
    "config.json", "model.safetensors.index.json", "tokenizer.json",
    "tokenizer_config.json", "outside.safetensors", "mtp.safetensors",
    "chat_template.jinja",
]
_missing = [name for name in _required if not (MODEL_PATH / name).is_file()]
if _missing:
    raise FileNotFoundError(f"Downloaded model is missing: {_missing}")

_layer_shards = sorted(MODEL_PATH.glob("model-layers-*.safetensors"))
_safetensors = sorted(MODEL_PATH.glob("*.safetensors"))
if len(_layer_shards) != 16 or len(_safetensors) != 18:
    raise RuntimeError(
        f"Unexpected checkpoint layout: {len(_layer_shards)} layer shards, "
        f"{len(_safetensors)} safetensors files (expected 16 / 18)."
    )
print(f"OK: {len(_safetensors)} safetensors files, {len(_layer_shards)} layer shards.")

In [ ]:
# Colab's own kernel is Python 3.13; vllm/flashinfer don't ship 3.13 wheels yet
# and ARC3-Inference pins requires-python==3.12.12. Create an isolated 3.12 venv
# with uv (downloads a standalone interpreter, no apt/sudo needed) and install
# everything into THAT instead -- we run vLLM and the harness through its
# binaries in every later cell, not through Colab's own Python.
!pip install -q uv

VENV_DIR = "/content/venv312"
!uv venv --python 3.12 {VENV_DIR}
VENV_PYTHON = f"{VENV_DIR}/bin/python"
!{VENV_PYTHON} --version

# torch + vllm from PyPI, pinned to match the real Kaggle deploy's stamp
# (atlas_src/setup_commands.json: 'vllm==0.19.0 torch==2.10.0 flashinfer==0.6.6').
!uv pip install --python {VENV_PYTHON} "torch==2.10.0" "vllm==0.19.0"

# flashinfer==0.6.6 from that stamp does NOT exist on public PyPI (confirmed --
# it's a private/custom build baked into the Kaggle wheelhouse, not a real
# PyPI release). Try the latest public flashinfer as a best-effort perf
# backend; if it fails, continue without it -- vLLM works with its default
# attention backend regardless, just possibly slower for this checkpoint.
!uv pip install --python {VENV_PYTHON} flashinfer || echo "flashinfer install failed -- continuing without it."

!uv pip install --python {VENV_PYTHON} -e "{ATLAS_SRC}/tufa-arc-agi-framework" -e "{ATLAS_SRC}/ARC3-Inference"

import subprocess

_check = subprocess.run(
    [VENV_PYTHON, "-c",
     "import importlib.util as u\n"
     "for m in ('vllm','torch','arc_agi','taaf','inference'):\n"
     "    print(m, '->', 'OK' if u.find_spec(m) else 'MISSING')"],
    capture_output=True, text=True,
)
print(_check.stdout)
if _check.returncode != 0:
    print(_check.stderr)

In [ ]:
# Start the vLLM OpenAI-compatible server, same flags as the real Kaggle deploy
# (atlas_src/setup_commands.json), pointed at the model we just downloaded.
# Runs through the 3.12 venv's python, not Colab's own kernel Python.
import json, os, subprocess, time
from pathlib import Path
from urllib.request import urlopen, Request

QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
VLLM_HOST, VLLM_PORT = "127.0.0.1", 1234
VLLM_BASE_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
VLLM_LOG = Path("/content/vllm-openai-server.log")

cmd = [
    VENV_PYTHON, "-m", "vllm.entrypoints.openai.api_server",
    "--model", str(MODEL_PATH),
    "--served-model-name", QWEN_SERVED_MODEL_NAME,
    "--host", VLLM_HOST, "--port", str(VLLM_PORT),
    "--tensor-parallel-size", "1",
    "--max-model-len", "65536",
    "--gpu-memory-utilization", "0.92",
    "--trust-remote-code",
    "--generation-config", "vllm",
    "--enable-prefix-caching",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    "--reasoning-parser", "qwen3",
    "--default-chat-template-kwargs", '{"preserve_thinking": true}',
]
print("Starting:", " ".join(cmd))
log_handle = VLLM_LOG.open("w", encoding="utf-8")
vllm_process = subprocess.Popen(cmd, stdout=log_handle, stderr=subprocess.STDOUT, text=True)
print(f"vLLM server PID: {vllm_process.pid} (log: {VLLM_LOG})")

def _request_json(url, payload=None, timeout=30):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = Request(url, data=data, headers={"Content-Type": "application/json"})
    with urlopen(req, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

deadline = time.monotonic() + 900
while time.monotonic() < deadline:
    if vllm_process.poll() is not None:
        print(VLLM_LOG.read_text(encoding="utf-8")[-4000:])
        raise RuntimeError(f"vLLM server exited early with code {vllm_process.returncode}")
    try:
        print("Server ready:", _request_json(f"{VLLM_BASE_URL}/models", timeout=5))
        break
    except Exception:
        time.sleep(5)
else:
    raise TimeoutError("Timed out waiting for vLLM server. Check the log:\n" + str(VLLM_LOG))

# Smoke test — a real generated reply, not just a health check.
smoke = _request_json(f"{VLLM_BASE_URL}/chat/completions", {
    "model": QWEN_SERVED_MODEL_NAME,
    "messages": [{"role": "user", "content": "Answer in one short sentence: what is 2 + 2?"}],
    "temperature": 0.0,
    "max_tokens": 96,
    "chat_template_kwargs": {"enable_thinking": False},
}, timeout=120)
print("Smoke test reply:", smoke["choices"][0]["message"].get("content", "").strip())

In [ ]:
# Analyzer/harness config, matching atlas_src/setup_commands.json and
# configs/inference.json exactly, so behavior is comparable to the real Kaggle run.
HARNESS_ENV = dict(os.environ)
HARNESS_ENV.update({
    "LOCAL_ANALYZER_BASE_URL": VLLM_BASE_URL,
    "OPENAI_BASE_URL": VLLM_BASE_URL,
    "LOCAL_ANALYZER_PROVIDER": "vllm",
    "OPENAI_PROVIDER": "vllm",
    "LOCAL_ANALYZER_API_KEY": "not-needed",  # the local server enforces no auth
    "LOCAL_ANALYZER_MODEL_ID": QWEN_SERVED_MODEL_NAME,
    "INFERENCE_ANALYZER_MODEL": QWEN_SERVED_MODEL_NAME,
    "LOCAL_ANALYZER_APP_NAME": "ARC3 Agent Harness",
    "LOCAL_ANALYZER_CONTEXT_WINDOW": "32768",
    # 8000, not the config default of 0/unbounded -- matches the atlas Kaggle
    # patch (see scripts/build_atlas_notebook.py, ATLAS_ANALYZER_MAX_OUTPUT_TOKENS)
    # so checkpoint/memo behavior is comparable to what v17-19 actually ran with.
    "LOCAL_ANALYZER_MAX_OUTPUT": "8000",
    "LOCAL_ANALYZER_TIMEOUT": "480",
    "ANALYZER_TIMEOUT": "480",
    "LOCAL_ANALYZER_TEMPERATURE": "0.6",
    "LOCAL_ANALYZER_TOP_P": "0.95",
    "LOCAL_ANALYZER_TOP_K": "20",
    "LOCAL_ANALYZER_TOOL_STEPS": "0",
    "LOCAL_ANALYZER_TOOL_TIMEOUT": "30",
    "LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS": "1024",
    "LOCAL_ANALYZER_YIELD_SECONDS": "60",
    "LOCAL_ANALYZER_ENABLE_THINKING": "true",
    "MULTIMODAL_CONTEXT": "current_grid",
    "MULTIMODAL_UPSCALE": "4",
    "HF_HUB_OFFLINE": "0",
    "TRANSFORMERS_OFFLINE": "0",
})
# ARC_API_KEY intentionally left unset -> arc_agi fetches an anonymous key for
# the ONLINE public API. Set HARNESS_ENV["ARC_API_KEY"] = "..." here if you
# have a real one and hit rate limits under anonymous access.
print("Harness env ready.")

## Run the 3 games

`CONCURRENT_JOBS = 1` runs them one at a time — cleanest transcripts, no GPU contention, easiest to actually read while it plays. Raise it to `len(GAME_IDS)` to run them in parallel and save wall-clock time, at the cost of interleaved logs and (per the retry-storm lesson from v17) some risk of analyzer contention if you push concurrency further than that.

`MAX_RUNTIME_MINUTES = 60` is generous on purpose — the whole point of this session is to *not* be quota-constrained. Raise it if a game is still visibly making progress when it hits the cap.

In [ ]:
import datetime, shlex

GAME_IDS = ["cn04-2fe56bfb", "vc33-5430563c", "ka59-38d34dbb"]
CONCURRENT_JOBS = 1
MAX_RUNTIME_MINUTES = 60
RUN_NAME = "colab-debug-" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
EXPERIMENT_DIR = f"/content/atlas_colab_runs/{RUN_NAME}"

cmd = [
    f"{VENV_DIR}/bin/inference-taaf-run",  # console script installed into the 3.12 venv
    "--game", ",".join(GAME_IDS),
    "--agent", "inference",
    "--model", QWEN_SERVED_MODEL_NAME,
    "--analyzer-timeout", "480",
    "--deployment-target", "inline",
    "--concurrent-jobs", str(CONCURRENT_JOBS),
    "--n-passes", "1",
    "--max-runtime-minutes", str(MAX_RUNTIME_MINUTES),
    "--run-name", RUN_NAME,
    "--experiment-dir", EXPERIMENT_DIR,
]
print("Running:", " ".join(shlex.quote(c) for c in cmd))
print(f"Transcripts will land in: {EXPERIMENT_DIR}\n")

process = subprocess.Popen(
    cmd, cwd=f"{ATLAS_SRC}/ARC3-Inference", env=HARNESS_ENV,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end="")
process.wait()
print(f"\ninference-taaf-run exited with code {process.returncode}")

In [ ]:
# Peek at what actually got written, and print each game's summary. The exact
# per-turn schema in *_events.jsonl varies with harness version -- print one
# raw event first so we can see the real keys before writing a deeper analysis
# (position-drift / memo-persistence / checkpoint-adoption) on top of it.
import json
from pathlib import Path

run_dir = Path(EXPERIMENT_DIR)
print("Files under", run_dir, ":")
for p in sorted(run_dir.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(run_dir), f"({p.stat().st_size} bytes)")

for viewer_data_path in sorted(run_dir.rglob("*_viewer_data.json")):
    data = json.loads(viewer_data_path.read_text(encoding="utf-8"))
    print(f"\n=== {viewer_data_path.name} (top-level keys) ===")
    print(list(data.keys()) if isinstance(data, dict) else type(data))

first_events = sorted(run_dir.rglob("*_events.jsonl"))
if first_events:
    lines = first_events[0].read_text(encoding="utf-8").splitlines()
    print(f"\n=== first event in {first_events[0].name} ===")
    if lines:
        print(json.dumps(json.loads(lines[0]), indent=2)[:2000])

## When you're done

Run the cell below to stop the vLLM server, then **Runtime -> Manage sessions -> Terminate** (or just close the tab) to stop the compute-unit meter -- it drains while the runtime is alive, not only while something is actively computing.

In [ ]:
vllm_process.terminate()
try:
    vllm_process.wait(timeout=30)
except Exception:
    vllm_process.kill()
print("vLLM server stopped.")